# ML Modelling: Training a Classifier You Can Actually Trust

**Dataset:** `sklearn.datasets.load_breast_cancer` — the Wisconsin Diagnostic Breast Cancer
dataset (569 real patient samples, 30 numeric features computed from digitized images of
fine-needle aspirate biopsies, binary target: malignant vs. benign).

**Problem statement.** A hospital wants a model that flags biopsy samples as *malignant* or
*benign* from the 30 measured features. Your job isn't just to train "a model that works" —
it's to build one whose reported quality you can actually stand behind. A single accuracy
number from one train/test split is not evidence of that: it can hide poor performance on the
class that matters most (missing a malignant tumor is far worse than a false alarm), and it can
swing wildly between runs if you don't control for variance.

**What you should produce**, concretely:
1. A `Pipeline` (`StandardScaler` + `LogisticRegression`) trained on a **stratified** train/test
   split, plus a `DummyClassifier` baseline to sanity-check that your model beats trivial
   guessing.
2. A full metrics report on the held-out test set: accuracy, precision, recall, F1, and ROC-AUC
   — reported **per class**, not just overall accuracy.
3. A 5-fold **stratified cross-validation** of F1 score (malignant class), reporting mean *and*
   standard deviation — demonstrating why one split's number isn't trustworthy on its own.
4. A `quality_gate(model, X, y, min_recall=0.95)` function that **asserts** recall on the
   malignant class meets a minimum threshold, raising if it doesn't — i.e., a model acceptance
   test, the same way you'd gate a build in SQA.
5. A short reproducibility check: train the same pipeline with two different `random_state`
   seeds and confirm the cross-validated F1 means don't drift by more than a small tolerance —
   flag it if they do.


---

## Starter cells

Fill in every `# TODO`. Don't look at the solution until you've tried each step.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import classification_report, roc_auc_score, f1_score

RANDOM_STATE = 42

data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="target")  # 0 = malignant, 1 = benign

print(X.shape)
print(y.value_counts(normalize=True))


In [ ]:
# TODO: split X, y into X_train, X_test, y_train, y_test
# - use test_size=0.25
# - use stratify=y so the class ratio is preserved in both splits
# - use random_state=RANDOM_STATE for reproducibility

X_train, X_test, y_train, y_test = ...  # TODO


In [ ]:
# TODO: build a baseline DummyClassifier (strategy="most_frequent") and a real pipeline.
#
# baseline: DummyClassifier(strategy="most_frequent")
# pipeline: Pipeline([("scaler", StandardScaler()), ("clf", LogisticRegression(max_iter=1000))])
#
# Fit both on X_train, y_train.

baseline = ...  # TODO
pipeline = ...  # TODO


In [ ]:
# TODO: on X_test, y_test, print:
#   - classification_report(y_test, y_pred) for the pipeline
#   - classification_report(y_test, baseline_pred) for the baseline
#   - roc_auc_score(y_test, y_proba) for the pipeline, where y_proba is predict_proba(...)[:, 1]
#
# Compare: does the pipeline beat the baseline on EVERY class, not just overall accuracy?


In [ ]:
# TODO: run 5-fold StratifiedKFold cross-validation of F1 score for the malignant class (label 0)
# on the FULL dataset (X, y) using the same pipeline structure.
#
# Hint: cross_val_score's `scoring` needs a scorer for the malignant class specifically —
# look at `make_scorer(f1_score, pos_label=0)` from sklearn.metrics.
#
# Report the mean and standard deviation of the 5 fold scores.


In [ ]:
# TODO: write quality_gate(model, X, y, min_recall=0.95) that:
#   - predicts on X
#   - computes recall for the malignant class (label 0)
#   - raises AssertionError with a clear message if recall < min_recall
#   - otherwise prints a pass message with the achieved recall
#
# Then call it on your fitted pipeline + X_test, y_test.

def quality_gate(model, X, y, min_recall=0.95):
    ...  # TODO


In [ ]:
# TODO: reproducibility check.
# Train the same pipeline structure with random_state=42 and random_state=123 (new splits each
# time), cross-validate F1 (malignant class) for each, and compare the two means.
# Flag (print a warning) if the difference exceeds 0.03 — that would mean your reported
# performance depends too much on the luck of the split/seed to be trustworthy.


---

## Solution

*(scroll down when you're ready — try it yourself first)*


<details>
<summary>Click to reveal solution</summary>

```python
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import classification_report, roc_auc_score, f1_score, recall_score, make_scorer

RANDOM_STATE = 42

data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="target")  # 0 = malignant, 1 = benign

# --- 1. stratified split ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE
)

baseline = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000)),
]).fit(X_train, y_train)

# --- 2. full metrics report, not just accuracy ---
y_pred = pipeline.predict(X_test)
y_proba = pipeline.predict_proba(X_test)[:, 1]
baseline_pred = baseline.predict(X_test)

print("=== Pipeline ===")
print(classification_report(y_test, y_pred, target_names=["malignant", "benign"]))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))

print("=== Baseline (most_frequent) ===")
print(classification_report(y_test, baseline_pred, target_names=["malignant", "benign"]))
# The baseline will show 0 recall/precision on malignant -- it never predicts that class.
# This is the "accuracy paradox": the baseline can still post decent overall accuracy
# (since benign is the majority class) while being useless for the class that matters.

# --- 3. cross-validated F1 on malignant class, mean + std ---
malignant_f1 = make_scorer(f1_score, pos_label=0)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_scores = cross_val_score(pipeline, X, y, cv=cv, scoring=malignant_f1)
print("CV F1 (malignant) scores:", cv_scores)
print(f"mean={cv_scores.mean():.4f}  std={cv_scores.std():.4f}")
# The std matters as much as the mean: a high mean with high std means the single
# number from one split could easily have looked a lot better or worse than reality.

# --- 4. quality gate, same spirit as an SQA acceptance test ---
def quality_gate(model, X, y, min_recall=0.95):
    y_pred = model.predict(X)
    recall_malignant = recall_score(y, y_pred, pos_label=0)
    assert recall_malignant >= min_recall, (
        f"Quality gate FAILED: malignant recall {recall_malignant:.3f} "
        f"is below required {min_recall}"
    )
    print(f"Quality gate PASSED: malignant recall = {recall_malignant:.3f} >= {min_recall}")

quality_gate(pipeline, X_test, y_test, min_recall=0.95)

# --- 5. reproducibility check across seeds ---
def cv_f1_for_seed(seed):
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, stratify=y, random_state=seed)
    pipe = Pipeline([("scaler", StandardScaler()), ("clf", LogisticRegression(max_iter=1000))])
    cv_seed = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    scores = cross_val_score(pipe.fit(Xtr, ytr), X, y, cv=cv_seed, scoring=malignant_f1)
    return scores.mean()

mean_42 = cv_f1_for_seed(42)
mean_123 = cv_f1_for_seed(123)
drift = abs(mean_42 - mean_123)
print(f"seed 42 mean F1={mean_42:.4f}  seed 123 mean F1={mean_123:.4f}  drift={drift:.4f}")
if drift > 0.03:
    print("WARNING: reported performance is seed-sensitive -- do not trust a single run's number.")
else:
    print("OK: performance is stable across seeds within tolerance.")
```

**Why this matters for ML quality engineering:** a model's "quality" is not one scalar from one
run. The dummy baseline exposes the accuracy paradox on classes that matter; the cross-validation
std exposes split-variance; the quality gate turns a soft preference ("recall should be high")
into a hard, automatable test exactly like a build gate in traditional SQA; and the seed-drift
check is a lightweight reproducibility test — the ML equivalent of a flaky-test detector.

</details>
